# Mutual Fund Performance Analytics
**Project:** Mutual Fund Analytics  
**Day:** 4 - Fund Performance Analytics  
**Purpose:** Quantify the historical performance, risk metrics (Sharpe, Sortino, Beta, Alpha, Drawdown), and track tracking errors for 40 mutual fund schemes vs. market benchmarks.  
**Prepared by:** Shree Aadharsh

In [ ]:
# 1. Import necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

# Set seaborn style for clean visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

In [ ]:
# 2. Load cleaned datasets
proc_dir = "../data/processed"
df_nav = pd.read_csv(f"{proc_dir}/nav_history_clean.csv")
df_fund = pd.read_csv(f"{proc_dir}/fund_master_clean.csv")
df_bench = pd.read_csv(f"{proc_dir}/benchmark_indices_clean.csv")

# Parse dates
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_bench['date'] = pd.to_datetime(df_bench['date'])
print("All datasets loaded successfully!")

---  
## 1. Daily Returns Calculation
We calculate daily returns for all 40 schemes using `nav_t / nav_t-1 - 1`. We'll also plot the daily returns distribution to verify that it is normally distributed with high concentration around 0%.

In [ ]:
df_nav_pivot = df_nav.pivot(index='date', columns='amfi_code', values='nav')
df_nav_returns = df_nav_pivot.pct_change()

# Plot the return distribution of a sample fund (e.g. SBI Bluechip Regular 119551)
plt.figure(figsize=(10, 5))
sns.histplot(df_nav_returns[119551].dropna(), bins=100, kde=True, color='teal')
plt.title('Daily Returns Distribution - SBI Bluechip Fund (Regular Plan)')
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
plt.xlim(-0.04, 0.04)
plt.show()

---  
## 2. CAGR Calculation (1-Year, 3-Year, & Since Inception)
We compute CAGR using the formula:
$$\text{CAGR} = \left( \frac{\text{NAV}_{\text{end}}}{\text{NAV}_{\text{start}}} \right)^{\frac{1}{n}} - 1$$
Since the daily historical data spans 4.4 years (Jan 2022 to May 2026), we will compute:  
1. **1-Year CAGR** (May 2025 - May 2026)  
2. **3-Year CAGR** (May 2023 - May 2026)  
3. **Inception CAGR** (Jan 2022 - May 2026, approx. 4.4 years) as a proxy for the 5-year CAGR requirement.

In [ ]:
end_date = df_nav_pivot.index.max()
date_1yr = end_date - pd.DateOffset(years=1)
date_3yr = end_date - pd.DateOffset(years=3)
date_start = df_nav_pivot.index.min()

def get_closest_nav(date_target, df_pivot):
    if date_target in df_pivot.index:
        return df_pivot.loc[date_target]
    idx = df_pivot.index.get_indexer([date_target], method='nearest')[0]
    return df_pivot.iloc[idx]

nav_end = df_nav_pivot.loc[end_date]
nav_1yr = get_closest_nav(date_1yr, df_nav_pivot)
nav_3yr = get_closest_nav(date_3yr, df_nav_pivot)
nav_start = df_nav_pivot.loc[date_start]

y_1yr = (end_date - date_1yr).days / 365.25
y_3yr = (end_date - date_3yr).days / 365.25
y_start = (end_date - date_start).days / 365.25

cagr_1yr = (nav_end / nav_1yr) ** (1.0 / y_1yr) - 1.0
cagr_3yr = (nav_end / nav_3yr) ** (1.0 / y_3yr) - 1.0
cagr_inception = (nav_end / nav_start) ** (1.0 / y_start) - 1.0

# Create comparison table
df_cagr = pd.DataFrame({
    'Scheme Name': [df_fund[df_fund['amfi_code'] == c]['scheme_name'].values[0] for c in df_nav_pivot.columns],
    '1Yr CAGR (%)': (cagr_1yr * 100).values,
    '3Yr CAGR (%)': (cagr_3yr * 100).values,
    'Inception CAGR (%)': (cagr_inception * 100).values
})
df_cagr.head(10)

---  
## 3. Sharpe & Sortino Ratios
We calculate the Sharpe and Sortino Ratios.  
- **Sharpe Ratio:** $\frac{R_p - R_f}{\sigma_{\text{annual}}}$  
- **Sortino Ratio:** $\frac{R_p - R_f}{\sigma_{\text{downside}}}$  
- We assume $R_f = 6.5\%$ (the RBI repo rate proxy).

In [ ]:
Rf = 0.065
ratios = []

for code in df_nav_pivot.columns:
    r_series = df_nav_returns[code].dropna()
    # Sharpe
    rp_annual = r_series.mean() * 252
    std_daily = r_series.std()
    std_annual = std_daily * np.sqrt(252)
    sharpe = (rp_annual - Rf) / std_annual if std_annual > 0 else 0
    
    # Sortino
    downside_returns = r_series[r_series < 0]
    downside_std_daily = np.sqrt(np.mean(downside_returns ** 2)) if len(downside_returns) > 0 else 0
    downside_std_annual = downside_std_daily * np.sqrt(252)
    sortino = (rp_annual - Rf) / downside_std_annual if downside_std_annual > 0 else 0
    
    ratios.append({
        'amfi_code': code,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino
    })

df_ratios = pd.DataFrame(ratios)
df_ratios_merge = pd.merge(df_cagr, df_ratios, left_on=df_cagr.index, right_on=df_ratios.index).drop(columns=['key_0'])
df_ratios_merge.head(10)

---  
## 4. Alpha & Beta Calculation
We regress each mutual fund's excess returns on Nifty 100 returns to compute:  
- **Beta (Market Sensitivity):** Slope of OLS regression.  
- **Alpha (Excess Outperformance):** Annualized intercept (Intercept $\times 252$).

In [ ]:
df_bench_pivot = df_bench.pivot(index='date', columns='index_name', values='close_value')
df_bench_returns = df_bench_pivot.pct_change()

ab_results = []
for code in df_nav_pivot.columns:
    r_series = df_nav_returns[code].dropna()
    df_align = pd.concat([r_series, df_bench_returns['NIFTY100']], axis=1, join='inner').dropna()
    slope, intercept, r_val, p_val, std_err = linregress(df_align['NIFTY100'], df_align[code])
    
    ab_results.append({
        'amfi_code': code,
        'Beta': slope,
        'Alpha (%)': intercept * 252 * 100.0
(

In [ ]:
df_ab = pd.DataFrame(ab_results)
df_ab_merge = pd.merge(df_ratios_merge, df_ab, on='amfi_code')
df_ab_merge.head(10)

---  
## 5. Maximum Drawdown & Date Ranges
Maximum Drawdown measures the worst peak-to-trough drop. We identify the exact dates of the peak and trough of this drawdown period.

In [ ]:
mdd_results = []
for code in df_nav_pivot.columns:
    nav_series = df_nav_pivot[code].dropna()
    running_max = nav_series.cummax()
    drawdowns = nav_series / running_max - 1.0
    max_dd = drawdowns.min()
    
    trough_idx = drawdowns.idxmin()
    peak_idx = nav_series.loc[:trough_idx].idxmax()
    
    mdd_results.append({
        'amfi_code': code,
        'Max Drawdown (%)': max_dd * 100.0,
        'Peak Date': peak_idx.strftime('%Y-%m-%d'),
        'Trough Date': trough_idx.strftime('%Y-%m-%d')
    })

df_mdd = pd.DataFrame(mdd_results)
df_perf_final = pd.merge(df_ab_merge, df_mdd, on='amfi_code')
df_perf_final.head(10)

---  
## 6. Composite Fund Scorecard (0-100)
We create a ranking score (0-100) using percentile ranks across five weighted metrics:
- **30%**: 3Yr CAGR Rank (higher is better)
- **25%**: Sharpe Ratio Rank (higher is better)
- **20%**: Alpha Rank (higher is better)
- **15%**: Expense Ratio Rank (inverse - lower is better)
- **10%**: Max Drawdown Rank (inverse - less negative is better)

This scorecard will allow us to objectively rank all 40 mutual fund schemes.

In [ ]:
# Fetch expense ratios
df_exp = df_fund[['amfi_code', 'expense_ratio_pct']]
df_score = pd.merge(df_perf_final, df_exp, on='amfi_code')

# Rank metrics as percentiles (0-100)
df_score['rank_return'] = df_score['3Yr CAGR (%)'].rank(pct=True) * 100.0
df_score['rank_sharpe'] = df_score['Sharpe Ratio'].rank(pct=True) * 100.0
df_score['rank_alpha']  = df_score['Alpha (%)'].rank(pct=True) * 100.0
df_score['rank_expense'] = (-df_score['expense_ratio_pct']).rank(pct=True) * 100.0
df_score['rank_drawdown'] = df_score['Max Drawdown (%)'].rank(pct=True) * 100.0

# Calculate composite score
df_score['Composite Score'] = (
    0.30 * df_score['rank_return'] +
    0.25 * df_score['rank_sharpe'] +
    0.20 * df_score['rank_alpha'] +
    0.15 * df_score['rank_expense'] +
    0.10 * df_score['rank_drawdown']
)

df_score = df_score.sort_values('Composite Score', ascending=False).reset_index(drop=True)
df_score['Rank'] = df_score.index + 1
df_score[['Rank', 'Scheme Name', 'Composite Score', '3Yr CAGR (%)', 'Sharpe Ratio', 'Alpha (%)', 'expense_ratio_pct', 'Max Drawdown (%)']].head(10)

---  
## 7. Benchmark Comparison & Tracking Error
We plot the top 5 funds vs. the `NIFTY50` and `NIFTY100` indices over the last 3 years and calculate the annualized Tracking Error vs. the Nifty 100 index.

In [ ]:
top_5_codes = df_score['amfi_code'].head(5).tolist()
three_yr_start = pd.Timestamp('2023-05-29')
df_nav_3y = df_nav_pivot.loc[three_yr_start:end_date]
df_bench_3y = df_bench_pivot.loc[three_yr_start:end_date]

# Normalize to base 100
df_nav_norm = (df_nav_3y / df_nav_3y.iloc[0]) * 100.0
df_bench_norm = (df_bench_3y / df_bench_3y.iloc[0]) * 100.0

plt.figure(figsize=(14, 7))
for code in top_5_codes:
    name = df_fund[df_fund['amfi_code'] == code]['scheme_name'].values[0].split(' - ')[0]
    plt.plot(df_nav_norm.index, df_nav_norm[code], label=name, linewidth=1.5)

plt.plot(df_bench_norm.index, df_bench_norm['NIFTY50'], label='NIFTY 50 (Benchmark)', color='black', linestyle='--', linewidth=2)
plt.plot(df_bench_norm.index, df_bench_norm['NIFTY100'], label='NIFTY 100 (Benchmark)', color='red', linestyle='-.', linewidth=2)

plt.title('Top 5 Mutual Funds vs Key Benchmarks - 3-Year Growth (Base 100)')
plt.xlabel('Date')
plt.ylabel('Normalized Value')
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
# Calculate Tracking Error
print("Annualized Tracking Error vs Nifty 100 (Last 3 Years):")
for code in top_5_codes:
    name = df_fund[df_fund['amfi_code'] == code]['scheme_name'].values[0]
    f_ret = df_nav_returns.loc[three_yr_start:end_date, code]
    b_ret = df_bench_returns.loc[three_yr_start:end_date, 'NIFTY100']
    df_align = pd.concat([f_ret, b_ret], axis=1).dropna()
    active_return = df_align[code] - df_align['NIFTY100']
    tracking_error = active_return.std() * np.sqrt(252) * 100.0
    print(f"- {name:<60}: {tracking_error:.2f}%")